In [16]:
import pandas as pd

df = pd.read_csv("juliet_cwe_sample.csv")

print("📦 Loaded Dataset:")
print(df.head(3))

print("\n🔢 Shape:", df.shape)
print("🔹 Unique CWEs:", df['cwe'].nunique())
print("🔹 Label counts:\n", df['label'].value_counts())

📦 Loaded Dataset:
                                           file_path  \
0  /Users/rohan/Documents/Final year/cwe_project/...   
1  /Users/rohan/Documents/Final year/cwe_project/...   
2  /Users/rohan/Documents/Final year/cwe_project/...   

                                            filename     cwe label  \
0  CWE114_Process_Control__w32_char_connect_socke...  CWE114  good   
1  CWE114_Process_Control__w32_wchar_t_console_83...  CWE114  good   
2  CWE114_Process_Control__w32_char_relativePath_...  CWE114  good   

                                            code_raw  \
0  /* TEMPLATE GENERATED TESTCASE FILE\nFilename:...   
1  /* TEMPLATE GENERATED TESTCASE FILE\nFilename:...   
2  /* TEMPLATE GENERATED TESTCASE FILE\nFilename:...   

                                          code_clean  cwe_index  label_id  
0  #include "std_testcase.h" #include <wchar.h> #...          0         0  
1  #ifndef OMITGOOD #include "std_testcase.h" #in...          0         0  
2  #include "std_testca

In [17]:
import re

def clean_code(code):
    if not isinstance(code, str):
        return ""
    code = re.sub(r"//.*", "", code)
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

df["code_clean"] = df["code_raw"].apply(clean_code)

print("\n🧹 RAW CODE SAMPLE (first 300 chars):")
print(df.iloc[0]["code_raw"][:300])

print("\n✨ CLEANED CODE SAMPLE (first 300 chars):")
print(df.iloc[0]["code_clean"][:300])


🧹 RAW CODE SAMPLE (first 300 chars):
/* TEMPLATE GENERATED TESTCASE FILE
Filename: CWE114_Process_Control__w32_char_connect_socket_11.c
Label Definition File: CWE114_Process_Control__w32.label.xml
Template File: sources-sink-11.tmpl.c
*/
/*
 * @description
 * CWE: 114 Process Control
 * BadSource: connect_socket Read data using a conne

✨ CLEANED CODE SAMPLE (first 300 chars):
#include "std_testcase.h" #include <wchar.h> #ifdef _WIN32 #include <winsock2.h> #include <windows.h> #include <direct.h> #pragma comment(lib, "ws2_32") #define CLOSE_SOCKET closesocket #else #include <sys/types.h> #include <sys/socket.h> #include <netinet/in.h> #include <arpa/inet.h> #include <unis


In [18]:
df["label_id"] = df["label"].map({"good": 0, "bad": 1})

print("\n🏷️ Encoded Labels Distribution:")
print(df["label_id"].value_counts())


🏷️ Encoded Labels Distribution:
label_id
1    1039
0    1026
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["cwe"],
    random_state=42
)

print("\n📘 Train size:", train_df.shape)
print("📙 Validation size:", val_df.shape)

print("\n🧪 Train unique CWEs:", train_df["cwe"].nunique())
print("🧪 Validation unique CWEs:", val_df["cwe"].nunique())


📘 Train size: (1652, 8)
📙 Validation size: (413, 8)

🧪 Train unique CWEs: 118
🧪 Validation unique CWEs: 118


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")

def tokenize_batch(batch):
    return tokenizer(
        batch["code_clean"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_batch(train_df)
val_encodings   = tokenize_batch(val_df)

print("\n🧩 Tokenization Complete!")
print("Train encodings keys:", train_encodings.keys())
print("Train input_ids shape:", train_encodings['input_ids'].shape)

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



🧩 Tokenization Complete!
Train encodings keys: dict_keys(['input_ids', 'attention_mask'])
Train input_ids shape: torch.Size([1652, 256])


In [21]:
sample_code = train_df.iloc[0]["code_clean"]

enc = tokenizer(sample_code, return_tensors="pt", truncation=True, padding="max_length", max_length=256)

print("\n🔹 Sample Clean Code:")
print(sample_code[:400], "...")

print("\n🔹 TOKENIZED INPUT IDS:")
print(enc["input_ids"])

print("\n🔹 ATTENTION MASK:")
print(enc["attention_mask"])


🔹 Sample Clean Code:
#include "std_testcase.h" #ifndef _WIN32 #include <wchar.h> #endif #ifdef _WIN32 #include <winsock2.h> #include <windows.h> #include <direct.h> #pragma comment(lib, "ws2_32") #define CLOSE_SOCKET closesocket #else #include <sys/types.h> #include <sys/socket.h> #include <netinet/in.h> #include <arpa/inet.h> #include <unistd.h> #define INVALID_SOCKET -1 #define SOCKET_ERROR -1 #define CLOSE_SOCKET c ...

🔹 TOKENIZED INPUT IDS:
tensor([[    1,     7,  6702,   315,  5084,    67,  3813,  3593,    18,    76,
             6,   468,   430,    82,   536,   389, 24572,  1578,   468,  6702,
           411,    91,  3001,    18,    76,    34,   468,   409,   430,   468,
           430,   536,   389, 24572,  1578,   468,  6702,   411,    91,  2679,
           975,    22,    18,    76,    34,   468,  6702,   411, 13226,    18,
            76,    34,   468,  6702,   411,  7205,    18,    76,    34,   468,
           683,  9454,  2879,    12,  2941,    16,   315,  4749,    22,    

In [22]:
train_encodings = tokenize_batch(train_df)
val_encodings   = tokenize_batch(val_df)

In [13]:
i = 0
print("Raw code:", train_df.iloc[i]["code_clean"][:300], "...")

enc = tokenizer(train_df.iloc[i]["code_clean"], return_tensors="pt")
print("\nInput IDs:", enc["input_ids"])
print("\nAttention mask:", enc["attention_mask"])

Token indices sequence length is longer than the specified maximum sequence length for this model (1056 > 512). Running this sequence through the model will result in indexing errors


Raw code: #include "std_testcase.h" #ifndef _WIN32 #include <wchar.h> #endif #ifdef _WIN32 #include <winsock2.h> #include <windows.h> #include <direct.h> #pragma comment(lib, "ws2_32") #define CLOSE_SOCKET closesocket #else #include <sys/types.h> #include <sys/socket.h> #include <netinet/in.h> #include <arpa/ ...

Input IDs: tensor([[   1,    7, 6702,  ...,  409,  430,    2]])

Attention mask: tensor([[1, 1, 1,  ..., 1, 1, 1]])


In [23]:
import torch
from torch.utils.data import Dataset

class JulietDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = JulietDataset(train_encodings, train_df["label_id"])
val_dataset   = JulietDataset(val_encodings,   val_df["label_id"])

print("\n📦 Dataset Ready!")
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


📦 Dataset Ready!
Train samples: 1652
Validation samples: 413


In [24]:
train_dataset = JulietDataset(train_encodings, train_df["label_id"])
val_dataset   = JulietDataset(val_encodings,   val_df["label_id"])